# 16 · Vecinas cercanas: resultados frente al V4

Protocolo: `docs/protocolo_vecinas_cercanas_v1.md` (`v4k_multianual_33_p95cal2023_v1`).
Idea del tesista: que cada detector use sólo las estaciones **geográficamente cercanas**,
para no recibir ruido de estaciones lejanas o de otro tipo de zona.

| Familia | Qué estaciones ve | Contexto |
|---|---|---|
| **V4** (referencia) | las 32 | t−24 … t−1 |
| **knn** | las k más cercanas, con k ∈ {4, 8, 16, 32} elegido con 2022 por estación | t−24 … t−1 |

Si para una estación gana k = 32, su detector **es** el V4 (se copia, no se reentrena).
Este libro **no entrena ni calcula**: lee lo que produce el script.

## Cómo ejecutar (desde la raíz del repo)

```bash
nohup .venv/bin/python experiments/vecinas/ejecutar.py > /dev/null 2>&1 &
tail -f results/vecinas_cercanas_v1/ejecucion.log
```

Reanudable: si se interrumpe, se relanza igual. Etapas: entrenar → verificar → evaluar →
simultáneo.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings('ignore')
from src.detect import multianual as m
from src.detect import vecinas as vc

K = RAIZ / 'results/vecinas_cercanas_v1'
V4 = RAIZ / 'results/lstm_v4_multianual_v1'
SIM_V4 = RAIZ / 'results/contemporaneo_v1/simultaneo/metricas_simultaneo_agregadas.csv'
IMG = RAIZ / 'docs/img/vecinas_cercanas_v1'
IMG.mkdir(parents=True, exist_ok=True)
FAM = {'v4': V4, 'knn': K}
COLOR = {'v4': '#52514e', 'knn': '#1baf7a'}
ETIQ = {'v4': 'V4 (32 vecinas)', 'knn': 'k vecinas cercanas'}
pd.set_option('display.width', 220)
plt.rcParams.update({'axes.grid': True, 'grid.color': '#e6e5e0', 'axes.spines.top': False,
                     'axes.spines.right': False, 'lines.linewidth': 2, 'legend.frameon': False,
                     'axes.titleweight': 'bold', 'axes.titlesize': 11})
cob = pd.read_csv(V4 / 'cobertura_etapas.csv')
CLASES = cob.drop_duplicates('objetivo').set_index('objetivo')['clasificacion']
geo = pd.read_csv(RAIZ / 'results/inventario_multianual/estaciones_geografia.csv', index_col=0)
DIST = vc.distancias(geo)
def leer(fam, archivo):
    f = FAM[fam] / archivo
    return pd.read_csv(f) if f.exists() else None

## 1. Estado de la ejecución

In [2]:
log = K / 'ejecucion.log'
print(''.join(log.read_text(encoding='utf-8').splitlines(keepends=True)[-8:]) if log.exists()
      else 'El script todavía no se ha ejecutado.')
n = len(list((K / 'estaciones').glob('*/configuracion.json'))) if (K / 'estaciones').exists() else 0
print(f'knn: {n}/31 estaciones listas')

El script todavía no se ha ejecutado.
knn: 0/31 estaciones listas


## 2. ¿Cuántas vecinas eligió cada estación?

`val_loss` es el error en 2022 (validación) con cada k; gana el menor. Si gana 32, la
estación se queda con el V4. La columna `dist_max_km` es qué tan lejos está la vecina más
lejana usada.

In [3]:
filas = []
for obj in m.ESTACIONES:
    f = K / 'estaciones' / obj / 'seleccion_k.csv'
    if f.exists():
        s = pd.read_csv(f).assign(objetivo=obj)
        s['elegido'] = s.k == json.loads((K / 'estaciones' / obj / 'configuracion.json').read_text())['k']
        filas.append(s)
if filas:
    SEL = pd.concat(filas, ignore_index=True)
    elegidos = SEL[SEL.elegido].set_index('objetivo')
    print('k elegido:', elegidos.k.value_counts().sort_index().to_dict())
    piv = SEL.pivot(index='objetivo', columns='k', values='val_loss_min').round(1)
    piv['k*'] = elegidos.k; piv['clasificación'] = CLASES.reindex(piv.index)
    piv['altitud_m'] = geo.loc[piv.index, 'alt']
    piv['dist_media_a_otras_km'] = DIST.loc[piv.index].replace(0, np.nan).mean(axis=1).round(1)
    print(piv.sort_values('k*').to_string())
else:
    print('Sin resultados de selección todavía.')

Sin resultados de selección todavía.


In [4]:
if filas:
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    ax = axes[0]
    for kk, color in zip([4, 8, 16, 32], ['#1baf7a', '#2a78d6', '#eda100', '#52514e']):
        d = geo.loc[elegidos.index[elegidos.k == kk]]
        ax.scatter(d.longitud, d.latitud, s=110, color=color, edgecolor='white', lw=1.2, zorder=3,
                   label=f'k = {kk}' + (' (V4)' if kk == 32 else ''))
    for s, r in geo.loc[elegidos.index].iterrows():
        ax.text(r.longitud + 0.006, r.latitud + 0.004, s, fontsize=7.5)
    ax.set_aspect(1 / np.cos(np.radians(19.4))); ax.legend(fontsize=8.5, loc='lower left')
    ax.set_title('k elegido por estación', loc='left')

    ax = axes[1]
    obj = 'CCA'
    if obj in elegidos.index and elegidos.at[obj, 'k'] < 32:
        usadas = elegidos.at[obj, 'vecinas'].split()
        otras = [s for s in m.ESTACIONES if s not in usadas + [obj]]
        ax.scatter(geo.loc[otras].longitud, geo.loc[otras].latitud, s=40, color='#c3c2b7', label='no usadas')
        ax.scatter(geo.loc[usadas].longitud, geo.loc[usadas].latitud, s=90, color='#1baf7a', label='usadas')
        ax.scatter(geo.at[obj, 'longitud'], geo.at[obj, 'latitud'], s=160, color='#e34948', marker='*', label=obj)
        for s in usadas + [obj]:
            ax.text(geo.at[s, 'longitud'] + 0.006, geo.at[s, 'latitud'] + 0.004, s, fontsize=8)
        ax.set_title(f'Vecinas que usa {obj} (k = {elegidos.at[obj, "k"]})', loc='left')
    else:
        ax.text(0.5, 0.5, f'{obj} eligió k = 32: usa todas (V4)', ha='center', transform=ax.transAxes)
        ax.set_title(f'Vecinas que usa {obj}', loc='left')
    ax.set_aspect(1 / np.cos(np.radians(19.4))); ax.legend(fontsize=8.5, loc='lower left')
    fig.tight_layout(); fig.savefig(IMG / 'k_elegido_mapa.png', dpi=150); plt.show()

## 3. Calibración 2023 (no es la prueba)

Error típico (MAE) y umbral p95 por estación. Sólo cambian las estaciones con k < 32.

In [5]:
def calibracion(fam):
    filas, base = [], FAM[fam] / 'estaciones'
    for obj in m.ESTACIONES:
        c = base / obj / 'calibracion_2023.csv.gz'
        if c.exists():
            cal = pd.read_csv(c)
            filas.append(dict(familia=fam, estacion=obj, clasificacion=CLASES[obj],
                              umbral=float((base / obj / 'umbral_p95_cal2023.txt').read_text()),
                              mae=cal.residuo_ppb.abs().mean(), mae_perfil=(cal.original_ppb - cal.base_ppb).abs().mean()))
    return pd.DataFrame(filas)
cal = pd.concat([calibracion(f) for f in FAM], ignore_index=True)
if cal.familia.nunique() > 1:
    print(cal.groupby(['familia', 'clasificacion'])[['umbral', 'mae', 'mae_perfil']].median().round(2).to_string())
    piv = cal.pivot(index='estacion', columns='familia', values='mae')
    piv['dif_mae'] = (piv.knn - piv.v4).round(2)
    print('\nEstaciones que cambiaron (k < 32):')
    print(piv[piv.dif_mae != 0].sort_values('dif_mae').round(2).to_string())
else:
    print('Aún no hay modelos knn.')

Aún no hay modelos knn.


## 4. Sin ataques: falsas alarmas en la prueba

In [6]:
if all(leer(f, 'metricas_limpias.csv') is not None for f in FAM):
    filas = []
    for fam in FAM:
        L = leer(fam, 'metricas_limpias.csv')
        for (anio, cl), g in L.groupby(['anio', 'clasificacion']):
            filas.append(dict(familia=fam, anio=anio, grupo=cl, mae_mediana=g.mae.median(),
                              fpr_micro_pct=100 * g.falsas_alarmas.sum() / g.n.sum()))
    print(pd.DataFrame(filas).pivot_table(index=['anio', 'grupo'], columns='familia',
                                          values=['fpr_micro_pct', 'mae_mediana']).round(2).to_string())
else:
    print('Falta la etapa «evaluar».')

Falta la etapa «evaluar».


## 5. Un mensaje atacado (muestra pareada al 5 %)

In [7]:
if all(leer(f, 'metricas_pareadas_agregadas.csv') is not None for f in FAM):
    A = pd.concat([leer(f, 'metricas_pareadas_agregadas.csv').assign(familia=f) for f in FAM])
    A = A[A.grupo == 'principal']
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    for ax, anio in zip(axes, [2024, 2025, 2026]):
        for fam in FAM:
            g = A[(A.familia == fam) & (A.anio == anio)]
            ax.plot(g.bit, g.recall_micro_pct, marker='o', color=COLOR[fam], label=ETIQ[fam])
        ax.set_title(f'{anio}' + (' (ene–jul)' if anio == 2026 else ''), loc='left')
        ax.set_xticks(range(8), [f'{b}\n±{2**b}' for b in range(8)]); ax.set_ylim(0, 103)
    axes[0].set_ylabel('% de ataques detectados'); axes[0].legend(fontsize=8.5)
    fig.tight_layout(); fig.savefig(IMG / 'recall_un_mensaje.png', dpi=150); plt.show()
    print(A.pivot_table(index=['anio', 'bit'], columns='familia',
                        values=['recall_micro_pct', 'precision_micro_pct', 'fpr_micro_pct']).round(1).to_string())
else:
    print('Falta la etapa «evaluar».')

Falta la etapa «evaluar».


## 6. Daño que se escapa y desactivaciones

In [8]:
if all(leer(f, 'resumen_dano_dias.csv') is not None for f in FAM):
    D = pd.concat([leer(f, 'resumen_dano_dias.csv').assign(familia=f) for f in FAM])
    print(D[D.tipo_dano.isin(['cambio_banda_red', 'falsa_fase1'])].pivot_table(
        index=['tipo_dano', 'anio', 'bit'], columns='familia', values='dias_dano_no_detectado').to_string())
    filas = []
    for fam in FAM:
        for anio in [2024, 2025, 2026]:
            f = FAM[fam] / f'eventos_dano/eventos_{anio}.parquet'
            if f.exists():
                ev = pd.read_parquet(f); an = ev[ev.anulada_fase1]
                exc = int(D[(D.familia == fam) & (D.anio == anio)].dias_con_excedencia_original.iat[0])
                filas.append(dict(familia=fam, anio=anio, dias_con_excedencia=exc, ocultables=an.fecha.nunique(),
                                  ocultables_sin_alerta=an[an.evaluable & ~an.detectado].fecha.nunique()))
    print('\nDesactivaciones (días):'); print(pd.DataFrame(filas).to_string(index=False))
else:
    print('Falta la etapa «evaluar».')

Falta la etapa «evaluar».


## 7. Atacante simultáneo

El V4 se toma de la ejecución de la variante contemporánea (mismo plan de horas, mismas
semillas).

In [9]:
fk = K / 'simultaneo/metricas_simultaneo_agregadas.csv'
if fk.exists() and SIM_V4.exists():
    S = pd.concat([pd.read_csv(SIM_V4).query("familia == 'v4'"), pd.read_csv(fk)])
    S = S[S.grupo == 'principal']
    print(S.pivot_table(index=['anio', 'bit'], columns='familia',
                        values=['recall_micro_pct', 'precision_micro_pct', 'fpr_micro_pct']).round(1).to_string())
else:
    print('Falta la etapa «simultaneo».')

Falta la etapa «simultaneo».


## 8. Notas

```
#### AAAA-MM-DD — título
- Qué se hizo:
- Resultados (k elegidos, calibración, limpio, un mensaje, daño, simultáneo):
- Interpretación:
- Decisiones / pendientes:
```